# RoadWise — Experiment C Training (2-Class Crack Detection)
### Production YOLOv8s Training on Google Colab (Tesla T4 GPU)

**Experiment Specs:**
- **Architecture:** YOLOv8s (`yolov8s.pt`)
- **Dataset:** Finalized Experiment C (2 classes: `0: longitudinal_crack`, `1: alligator_crack`)
- **Dataset Composition:**
  - Train: **2,918 images** (5,678 boxes: D00=2,051, D20=3,627)
  - Validation: **339 images** (683 boxes: D00=277, D20=406)
  - Test: **335 images** (601 boxes: D00=240, D20=361 — held-out test split)
- **Hyperparameters:**
  - `epochs`: 100
  - `imgsz`: 640
  - `batch`: 16 (or auto)
  - `fliplr`: 0.0 (preserves McMaster flip augmentation geometry)
  - `device`: 0 (GPU)
  - `patience`: 30 (early stopping)
  - `save_period`: 10
- **Motivation:** Experiment C isolates D00 vs D20 (removes D10) to test whether removing transverse crack class improves D20 recall from the Experiment A baseline of 14.7%.


## 1. Verify GPU Hardware
Check that a GPU runtime (Tesla T4, V100, or A100) is connected.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CRITICAL ERROR: No GPU found. Go to Runtime -> Change runtime type -> Select T4 GPU!'
print(f'\u2705 Active GPU: {torch.cuda.get_device_name(0)}')
print(f'\u2705 CUDA Version: {torch.version.cuda}')

## 2. Install Dependencies
Install official Ultralytics YOLO and supporting packages.

In [ ]:
!pip install -q ultralytics pyyaml pillow
import ultralytics
ultralytics.checks()

## 3. Mount Google Drive & Initialize Versioned Backup Directory
Mount Google Drive to access `crack_experiment_c_2class.zip` and initialize a unique, non-destructive versioned run directory (`yolov8s_2class_final_run_YYYYMMDD_HHMMSS`) to safely preserve all checkpoints without overwriting prior training runs.

In [ ]:
from google.colab import drive
import os, shutil, zipfile
from datetime import datetime

drive.mount('/content/drive')

# Base Drive directory where training runs are organized
DRIVE_RUNS_DIR = '/content/drive/MyDrive/RoadWise_Training'
assert os.path.exists('/content/drive/MyDrive'), 'CRITICAL ERROR: Google Drive /content/drive/MyDrive is not mounted or inaccessible!'
os.makedirs(DRIVE_RUNS_DIR, exist_ok=True)

# Non-destructive versioned destination for this training session
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_NAME = f'yolov8s_2class_final_run_{RUN_TIMESTAMP}'
TARGET_DRIVE_DIR = os.path.join(DRIVE_RUNS_DIR, RUN_NAME)
os.makedirs(TARGET_DRIVE_DIR, exist_ok=False)  # Guarantees unique fresh directory; never overwrites
os.makedirs(os.path.join(TARGET_DRIVE_DIR, 'weights'), exist_ok=True)

# Persist target directory path locally for reliable retrieval across cells
with open('/content/roadwise_c_run_destination.txt', 'w') as f:
    f.write(TARGET_DRIVE_DIR)

print(f'\u2705 Drive base directory:         {DRIVE_RUNS_DIR}')
print(f'\u2705 Initialized run destination:   {TARGET_DRIVE_DIR}')

## 4. Extract Dataset to `/content/` for Fast NVMe I/O
Locate `crack_experiment_c_2class.zip` (from Drive or local upload) and unpack directly into the fast local VM filesystem.

In [ ]:
# Path options for the zip file:
# Option A: Uploaded directly to Colab
# Option B: Stored on Google Drive (e.g. MyDrive/RoadWise/crack_experiment_c_2class.zip)
zip_candidates = [
    '/content/crack_experiment_c_2class.zip',
    '/content/drive/MyDrive/RoadWise/crack_experiment_c_2class.zip',
    '/content/drive/MyDrive/RoadWise_Training/crack_experiment_c_2class.zip',
    '/content/drive/MyDrive/crack_experiment_c_2class.zip'
]

zip_path = None
for cand in zip_candidates:
    if os.path.exists(cand):
        zip_path = cand
        break

if zip_path is None:
    print('\u26a0\ufe0f Please upload crack_experiment_c_2class.zip directly using the Colab file explorer on the left or place it in MyDrive!')
    from google.colab import files
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]

print(f'Extracting dataset from {zip_path} ...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')

# Standardize directory path
if not os.path.exists('/content/crack_experiment_c_2class'):
    for root, dirs, files in os.walk('/content'):
        if 'crack_experiment_c_2class' in dirs:
            shutil.move(os.path.join(root, 'crack_experiment_c_2class'), '/content/crack_experiment_c_2class')
            break

print('\u2705 Dataset unpacked successfully to /content/crack_experiment_c_2class')

## 5. Verify Dataset Integrity & Generate Colab `data.yaml`
Assert exact image counts (Train=2918, Val=339, Test=335) and write `/content/crack_experiment_c_2class/data_colab.yaml`.

In [ ]:
DATASET_DIR = '/content/crack_experiment_c_2class'

# Count images in each split
train_imgs = len(os.listdir(f'{DATASET_DIR}/train/images'))
val_imgs = len(os.listdir(f'{DATASET_DIR}/val/images'))
test_imgs = len(os.listdir(f'{DATASET_DIR}/test/images'))

print(f'Train images: {train_imgs} (Expected: 2918)')
print(f'Val images:   {val_imgs} (Expected: 339)')
print(f'Test images:  {test_imgs} (Expected: 335)')

assert train_imgs == 2918, f'Train count mismatch: {train_imgs} != 2918'
assert val_imgs == 339, f'Val count mismatch: {val_imgs} != 339'
assert test_imgs == 335, f'Test count mismatch: {test_imgs} != 335'

# Write local data_colab.yaml
yaml_content = f"""# RoadWise Experiment C — Colab Configuration
path: {DATASET_DIR}
train: train/images
val: val/images
test: test/images

nc: 2
names:
  0: longitudinal_crack
  1: alligator_crack
"""

COLAB_YAML = f'{DATASET_DIR}/data_colab.yaml'
with open(COLAB_YAML, 'w') as f:
    f.write(yaml_content)

print(f'\u2705 Verified dataset integrity and written: {COLAB_YAML}')

## 6. Execute Experiment C Model Training with Periodic Checkpoint Backup
Train YOLOv8s for 100 epochs at 640x640 with `fliplr=0.0`. A lightweight background daemon thread periodically mirrors newly created checkpoints (`weights/last.pt`, `weights/best.pt`, periodic `epoch*.pt`) and training metrics (`results.csv`, plots) directly to Google Drive via atomic writes without interrupting model training.

In [ ]:
from ultralytics import YOLO
import time
import os
import shutil
import threading

# Retrieve versioned backup destination initialized in Cell 3
if 'TARGET_DRIVE_DIR' not in globals() or not TARGET_DRIVE_DIR:
    if os.path.exists('/content/roadwise_c_run_destination.txt'):
        with open('/content/roadwise_c_run_destination.txt', 'r') as f:
            TARGET_DRIVE_DIR = f.read().strip()

assert 'TARGET_DRIVE_DIR' in globals() and TARGET_DRIVE_DIR and os.path.exists(TARGET_DRIVE_DIR), (
    f"CRITICAL ERROR: Versioned destination '{globals().get('TARGET_DRIVE_DIR')}' is unavailable. "
    "Please run Cell 3 to mount Drive and initialize the destination before training."
)

TRAIN_RUN_DIR = '/content/runs/roadwise_experiment_c/yolov8s_2class'

def safe_atomic_copy(src_path, dst_path):
    """
    Copies src_path to dst_path safely by writing to a temporary file first
    and atomically replacing dst_path, preventing partial/corrupted files in Google Drive.
    """
    if not os.path.exists(src_path):
        return False
    src_size = os.path.getsize(src_path)
    if src_size == 0:
        return False  # Wait for file to have content

    dst_dir = os.path.dirname(dst_path)
    os.makedirs(dst_dir, exist_ok=True)
    temp_dst = dst_path + f'.tmp_{int(time.time() * 1000)}'

    try:
        shutil.copy2(src_path, temp_dst)
        if os.path.getsize(temp_dst) != src_size:
            if os.path.exists(temp_dst):
                os.remove(temp_dst)
            return False
        os.replace(temp_dst, dst_path)
        return True
    except Exception as e:
        if os.path.exists(temp_dst):
            try:
                os.remove(temp_dst)
            except Exception:
                pass
        raise e

# Lightweight background thread: periodically mirrors checkpoints & results to Google Drive
class PeriodicCheckpointBackup(threading.Thread):
    def __init__(self, src_dir, dst_dir, interval_sec=60):
        super().__init__(daemon=True)
        self.src_dir = src_dir
        self.dst_dir = dst_dir
        self.interval_sec = interval_sec
        self.stop_event = threading.Event()
        self.final_sync_done = threading.Event()
        self.synced_mtimes = {}
        self.sync_errors = []

    def sync_files(self, is_final=False):
        errors = []
        try:
            if not os.path.exists(self.src_dir):
                return errors

            # 1. Sync weights (last.pt, best.pt, periodic epoch*.pt)
            src_weights = os.path.join(self.src_dir, 'weights')
            dst_weights = os.path.join(self.dst_dir, 'weights')
            if os.path.exists(src_weights):
                for fname in sorted(os.listdir(src_weights)):
                    if fname.endswith('.pt'):
                        s_file = os.path.join(src_weights, fname)
                        d_file = os.path.join(dst_weights, fname)
                        try:
                            mtime = os.path.getmtime(s_file)
                            if is_final or fname not in self.synced_mtimes or mtime > self.synced_mtimes[fname]:
                                if safe_atomic_copy(s_file, d_file):
                                    self.synced_mtimes[fname] = mtime
                        except Exception as e:
                            errors.append(f'Checkpoint copy failed ({fname}): {e}')

            # 2. Sync training metrics, plots, and configs
            for fname in sorted(os.listdir(self.src_dir)):
                if fname.endswith(('.csv', '.png', '.jpg', '.jpeg', '.yaml')):
                    s_file = os.path.join(self.src_dir, fname)
                    if os.path.isfile(s_file):
                        d_file = os.path.join(self.dst_dir, fname)
                        try:
                            mtime = os.path.getmtime(s_file)
                            if is_final or fname not in self.synced_mtimes or mtime > self.synced_mtimes[fname]:
                                if safe_atomic_copy(s_file, d_file):
                                    self.synced_mtimes[fname] = mtime
                        except Exception as e:
                            errors.append(f'Metric/plot copy failed ({fname}): {e}')

        except Exception as e:
            errors.append(f'Sync exception: {e}')

        if errors:
            self.sync_errors.extend(errors)
        return errors

    def run(self):
        while not self.stop_event.is_set():
            self.sync_files(is_final=False)
            self.stop_event.wait(self.interval_sec)
        # Execute comprehensive final sync when stopped
        self.sync_files(is_final=True)
        self.final_sync_done.set()

    def stop_and_wait(self):
        """
        Stops the backup thread and blocks until the final sync has completely finished.
        """
        self.stop_event.set()
        self.join()  # Proper completion: waits for thread termination
        self.final_sync_done.wait()
        return self.sync_errors

# Start lightweight periodic backup
backup_thread = PeriodicCheckpointBackup(TRAIN_RUN_DIR, TARGET_DRIVE_DIR, interval_sec=60)
backup_thread.start()
print(f'\u2705 Periodic checkpoint backup running (60s interval) -> {TARGET_DRIVE_DIR}')

# Load pretrained YOLOv8s weights
model = YOLO('yolov8s.pt')

start_time = time.time()
print('====================================================')
print('STARTING EXPERIMENT C MODEL TRAINING')
print('====================================================')

try:
    results = model.train(
        data=COLAB_YAML,
        epochs=100,
        imgsz=640,
        batch=16,
        fliplr=0.0,       # Strict requirement: preserves McMaster flip copies
        device=0,
        project='/content/runs/roadwise_experiment_c',
        name='yolov8s_2class',
        save=True,
        save_period=10,
        plots=True,
        val=True,
        workers=4,
        patience=30
    )
finally:
    print('\nFinalizing training and completing checkpoint synchronization...')
    sync_errors = backup_thread.stop_and_wait()
    
    # Check physical presence of essential checkpoints at destination
    weights_dst = os.path.join(TARGET_DRIVE_DIR, 'weights')
    expected_last = os.path.join(weights_dst, 'last.pt')
    expected_best = os.path.join(weights_dst, 'best.pt')
    results_dst = os.path.join(TARGET_DRIVE_DIR, 'results.csv')

    if sync_errors:
        print('\u26a0\ufe0f The following synchronization issues occurred during training:')
        for err in sync_errors[-10:]:
            print(f'   - {err}')

    # Fail clearly if essential files were not backed up
    assert os.path.exists(TARGET_DRIVE_DIR), f'CRITICAL BACKUP FAILURE: Destination {TARGET_DRIVE_DIR} does not exist!'
    assert os.path.exists(expected_last), f'CRITICAL BACKUP FAILURE: last.pt was not backed up to {expected_last}!'
    assert os.path.exists(results_dst), f'CRITICAL BACKUP FAILURE: results.csv was not backed up to {results_dst}!'
    if os.path.exists(os.path.join(TRAIN_RUN_DIR, 'weights', 'best.pt')):
        assert os.path.exists(expected_best), f'CRITICAL BACKUP FAILURE: best.pt exists locally but was not backed up to {expected_best}!'

    print(f'\u2705 All checkpoints and results successfully synchronized to: {TARGET_DRIVE_DIR}')

train_duration = (time.time() - start_time) / 3600.0
print(f'\u2705 Training completed in {train_duration:.2f} hours!')

## 7. Evaluate on Held-Out Test Split (Benchmark Evaluation)
Evaluate the newly trained `best.pt` on the completely untouched Test split (335 images: D00=240, D20=361).

> **Key comparison metric:** D20 recall vs Experiment A baseline (14.7%). If D20 recall > 50%, consider Soft-NMS and imgsz=800 for future work.

In [ ]:
best_checkpoint = '/content/runs/roadwise_experiment_c/yolov8s_2class/weights/best.pt'
assert os.path.exists(best_checkpoint), f'Checkpoint not found: {best_checkpoint}'

eval_model = YOLO(best_checkpoint)
eval_model.add_callback = getattr(eval_model, 'add_callback', lambda *args, **kwargs: None)

print('Evaluating on held-out Test split (335 images)...')
test_results = eval_model.val(
    data=COLAB_YAML,
    split='test',
    imgsz=640,
    device=0,
    plots=True
)

print('\n====================================================')
print('FINAL HELD-OUT TEST SPLIT EVALUATION METRICS')
print('====================================================')
print(f'Overall mAP@50:    {test_results.box.map50:.4f}')
print(f'Overall mAP@50-95: {test_results.box.map:.4f}')
print(f'Precision:         {test_results.box.p.mean():.4f}')
print(f'Recall:            {test_results.box.r.mean():.4f}')

# Per-class results
class_names = ['longitudinal_crack (D00)', 'alligator_crack (D20)']
print('\nPer-Class Breakdown:')
for i, cname in enumerate(class_names):
    if i < len(test_results.box.maps):
        print(f'  \u2022 {cname:26}: mAP@50={test_results.box.ap50[i]:.4f}, mAP@50-95={test_results.box.maps[i]:.4f}')

# Compare against Experiment A D20 baseline
EXP_A_D20_RECALL = 0.147
EXP_A_D20_MAP50 = 0.1563
print('\n--- Experiment C vs Experiment A D20 Baseline ---')
if len(test_results.box.r) > 1:
    exp_c_d20_recall = float(test_results.box.r[1])
    exp_c_d20_map50 = float(test_results.box.ap50[1])
    recall_delta = exp_c_d20_recall - EXP_A_D20_RECALL
    map_delta = exp_c_d20_map50 - EXP_A_D20_MAP50
    print(f'Exp A D20 Recall: {EXP_A_D20_RECALL:.3f}  |  Exp C D20 Recall: {exp_c_d20_recall:.3f}  |  Delta: {recall_delta:+.3f}')
    print(f'Exp A D20 mAP50:  {EXP_A_D20_MAP50:.4f} |  Exp C D20 mAP50:  {exp_c_d20_map50:.4f} |  Delta: {map_delta:+.4f}')
    if exp_c_d20_recall > 0.50:
        print('\u2705 D20 recall > 50%: Consider Soft-NMS and imgsz=800 for future experiments.')
    elif exp_c_d20_recall < 0.30:
        print('\u26a0\ufe0f D20 recall < 30%: Investigate annotation clustering and box merging strategies.')
    else:
        print(f'\u2139\ufe0f D20 recall {exp_c_d20_recall:.1%}: Moderate improvement. Assess annotation strategy for next steps.')

## 8. Non-Destructive Final Artifact & Evaluation Sync to Google Drive
Performs a non-destructive final synchronization of all artifacts, evaluation plots, and checkpoints to the versioned Google Drive run directory without deleting previous runs.

In [ ]:
import os
import shutil

# Resolve destination directory strictly from initialization; never fallback to an unversioned directory
if 'TARGET_DRIVE_DIR' not in globals() or not TARGET_DRIVE_DIR:
    if os.path.exists('/content/roadwise_c_run_destination.txt'):
        with open('/content/roadwise_c_run_destination.txt', 'r') as f:
            TARGET_DRIVE_DIR = f.read().strip()

assert 'TARGET_DRIVE_DIR' in globals() and TARGET_DRIVE_DIR and os.path.exists(TARGET_DRIVE_DIR), (
    f"CRITICAL ERROR: Versioned destination directory '{globals().get('TARGET_DRIVE_DIR')}' is not available. "
    "Drive may have become unmounted. Halting to prevent data loss."
)

source_run_dir = '/content/runs/roadwise_experiment_c/yolov8s_2class'
assert os.path.exists(source_run_dir), f'CRITICAL ERROR: Training run directory {source_run_dir} not found!'

print(f'Final non-destructive synchronization to: {TARGET_DRIVE_DIR} ...')
shutil.copytree(source_run_dir, TARGET_DRIVE_DIR, dirs_exist_ok=True)

# Also sync test evaluation artifacts if saved under /content/runs/roadwise_experiment_c
test_runs_dir = '/content/runs/roadwise_experiment_c'
if os.path.exists(test_runs_dir):
    for item in sorted(os.listdir(test_runs_dir)):
        if item != 'yolov8s_2class':
            src_item = os.path.join(test_runs_dir, item)
            dst_item = os.path.join(TARGET_DRIVE_DIR, item)
            if os.path.isdir(src_item):
                shutil.copytree(src_item, dst_item, dirs_exist_ok=True)

# Final audit of preserved artifacts
print('\n\u2705 Artifacts safely preserved in Google Drive:')
artifact_count = 0
for root, dirs, files in os.walk(TARGET_DRIVE_DIR):
    for f in sorted(files):
        artifact_count += 1
        print('  \u2022', os.path.relpath(os.path.join(root, f), TARGET_DRIVE_DIR))

assert artifact_count > 0, f'CRITICAL ERROR: No artifacts found in {TARGET_DRIVE_DIR}!'
assert os.path.exists(os.path.join(TARGET_DRIVE_DIR, 'weights', 'last.pt')), 'CRITICAL ERROR: last.pt missing from final backup!'

print(f'\n\u2705 Versioned run directory: {TARGET_DRIVE_DIR}')
print(f'\u2705 Best model checkpoint:   {os.path.join(TARGET_DRIVE_DIR, "weights", "best.pt")}')
print(f'\u2705 Last model checkpoint:   {os.path.join(TARGET_DRIVE_DIR, "weights", "last.pt")}')
print(f'\u2705 Training metrics CSV:     {os.path.join(TARGET_DRIVE_DIR, "results.csv")}')
print(f'\u2705 Total preserved files:   {artifact_count}')